In [3]:
import torch
import torchvision
from pathlib import Path
import pandas as pd

In [4]:
device = torch.device("cpu")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

In [5]:
BASE_DIR = Path.cwd().parent.parent
TRAIN_DIR = Path(BASE_DIR/'data', 'images', 'train')
VAL_DIR = Path(BASE_DIR/'data', 'images', 'val')
TEST_DIR = Path(BASE_DIR/'data', 'images', 'test')

SPLIT_CSV_PATH = Path(BASE_DIR/'data', 'split.csv')

df = pd.read_csv(SPLIT_CSV_PATH)
df



,filepath,category,defect_type,split
0,test/good/000.png,OK,good,train
1,test/good/001.png,OK,good,train
2,test/good/002.png,OK,good,train
3,test/good/003.png,OK,good,train
4,test/good/004.png,OK,good,test
...,...,...,...,...
475,train/good/315.png,OK,good,train
476,train/good/316.png,OK,good,train
477,train/good/317.png,OK,good,train
478,train/good/318.png,OK,good,test


In [6]:
import sys
sys.path.append(str(BASE_DIR / "src"))
import utils
from utils import ScrewDataset

In [7]:
df['new_filepath'] = df['filepath'].str.replace(
    r'^([^/]+)/([^/]+)/', r'\1_\2__', regex=True)

In [8]:
cols = ['new_filepath', 'category', 'defect_type']

train_manifest = df.loc[df['split']=='train'][cols]
val_manifest = df.loc[df['split']=='val'][cols]
test_manifest = df.loc[df['split']=='test'][cols]

In [9]:
eval_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [10]:
test_dataset = ScrewDataset(test_manifest, TEST_DIR, eval_transform)

In [11]:
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

In [12]:
model = utils.build_model_resnet18(2)
model = model.to(device)

In [13]:
state_dict = torch.load(r'C:\Users\maciej.klimczak\maciej_klimczak\studia\inzynierka\models\resnet18_baseline\best.pth', map_location=device)
model.load_state_dict(state_dict)

C:\Users\maciej.klimczak\AppData\Local\Temp\ipykernel_42652\122217616.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(r'C:\Users\maciej.klimczak\

<All keys matched successfully>

In [14]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["OK", "NOK"]))

[[24 30]
 [ 0 18]]
              precision    recall  f1-score   support

          OK       1.00      0.44      0.62        54
         NOK       0.38      1.00      0.55        18

    accuracy                           0.58        72
   macro avg       0.69      0.72      0.58        72
weighted avg       0.84      0.58      0.60        72

